In [1]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import plotly.express as px
import numpy as np

sys.path.append(os.path.join("..", "scripts", "analysis"))
from processing import read_KPI_adequacy, apply_conservative_classification, load_solutions, combine_solutions

G_save = False

# dim = (1000,500)
# g_BLUE = "1616A7"
# g_GREY = "#7F7F7F"
g_ORANGE = "Orange"



In [2]:
latex_textwidth_pt = 516.0 # double column # use this command in latex: \the\textwidth
# latex_textwidth_pt = 452.0 # single column
scale = 1
dim = (latex_textwidth_pt * scale,latex_textwidth_pt * scale*0.6)

legend_attr = dict(
    x=0.5,
    y=-0.25,
    yanchor="bottom",
    xanchor="center",
    orientation="h"
)

def update_background(fig, legend_attr=legend_attr, dim=dim, showlegend=True):
    fig.update_layout(
        plot_bgcolor="rgba(0,0,0,0)",
        # yaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True),
        # xaxis=dict(showgrid=False, gridwidth=0.11, gridcolor="grey", showline=True, linecolor="grey", mirror=True),
        width=dim[0],
        height=dim[1],
        showlegend=showlegend,
        autosize=False,
        legend=legend_attr,  # Include legend attributes
        # font=dict(
        #     family="Computer Modern",  # or 'Arial', 'Courier New', etc.
        #     # size=12,                   # default font size for all text
        # #     # color="black"              # font color
        #     )
    )
    config = dict(showgrid=False, showticklabels=True, showline=True, linecolor="grey", mirror=True, gridwidth=0.11, gridcolor="grey")
    # fig.for_each_yaxis(lambda yaxis: yaxis.update(**config))
    # fig.for_each_xaxis(lambda xaxis: xaxis.update(**config))
    fig.update_yaxes(**config)
    
    fig.update_xaxes(**config)
    
    fig.update_traces(
        boxmean=True,
        selector=dict(type='box')
    )
    # for axis in fig.layout:
    #     if axis.startswith('xaxis') or axis.startswith('yaxis'):
    #         fig.layout[axis].update(showticklabels=True, linecolor="grey", mirror=True)

    # for axis in fig.layout:
    #     if axis.startswith('xaxis') or axis.startswith('yaxis'):
    #         fig.layout[axis].update(title = '')


In [3]:
def add_up_down_multiindex(df):
    new_columns = []
    for col in df.columns:
        col_str = str(col)
        if 'up' in col_str:
            # Remove 'up_' or '_up' from the column name
            if 'up_' in col_str:
                base_col = col_str.replace('up_', '')
            else:
                base_col = col_str
            new_columns.append(('up', base_col))
        elif 'down' in col_str:
            # Remove 'down_' or '_down' from the column name
            if 'down_' in col_str:
                base_col = col_str.replace('down_', '')
            else:
                base_col = col_str
            new_columns.append(('down', base_col))
        else:
            new_columns.append(('', col))
    df.columns = pd.MultiIndex.from_tuples(new_columns)
    df = df.stack(level=0, future_stack=True)
    df.index = df.index.set_names('reserve_direction', level=-1)
    return df

In [4]:

ss = [
    {'solution_folder': f"RTS-GMLC_v32.3s", 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v32.1s", 'model_type' : 'e-reserve'},
    ]




Data construction and filtering

In [5]:
import gc

days = range(1,365)
s_uc = []
s_ed = []
solution_keys = ['reserve','energy_reserve', 'storage']

# Only load columns used by this notebook to reduce memory footprint.
columns_by_key = {
    'reserve': [
        'hour', 'day', 'resource', 'µ', 'model_type','iteration',
        'reserve_up_MW', 'reserve_down_MW',
        'required_reserve_up_MW', 'required_reserve_down_MW',
        'configuration'
    ],
    'energy_reserve': [
        'hour', 'hour_i', 'day', 'resource', 'model_type', 'iteration',
        'energy_reserve_up_MW', 'energy_reserve_down_MW',
        'required_energy_reserve_up_MW', 'required_energy_reserve_down_MW',
        'configuration'
    ],
    'storage': [
        'day', 'hour', 'resource', 'r_id', 'model_type', 'µ', 'iteration',
        'charge_MW', 'discharge_MW', 'scenario', 'configuration'
    ]
}

for sol in ss:
    s = sol['solution_folder']
    s_uc_ = load_solutions(
        "s_uc",
        os.path.join("..", "output", s),
        days,
        solution_keys=solution_keys,
        columns_by_key=columns_by_key,
        model_type=sol['model_type'],
        solution_id=s,
    )
    if sol['model_type'] != 'stochastic':
        s_ed_ = load_solutions(
            "s_ed",
            os.path.join("..", "output", s),
            days,
            solution_keys=solution_keys,
            columns_by_key=columns_by_key,
            model_type=sol['model_type'],
            solution_id=s,
        )
    else:
        s_ed_ = load_solutions(
            "s_suc",
            os.path.join("..", "output", s),
            days,
            solution_keys=solution_keys,
            columns_by_key=columns_by_key,
            model_type=sol['model_type'],
            solution_id=s,
        )
    s_uc.append(s_uc_)
    s_ed.append(s_ed_)

s_uc = combine_solutions(s_uc)
s_ed = combine_solutions(s_ed)

del s_uc_, s_ed_
gc.collect()

0

In [ ]:

s_storage_e_reserve = s_uc['energy_reserve'][(s_uc['energy_reserve'].resource.isin(['battery','hydro_reservoir', 'CSP']))&(s_uc['energy_reserve'].model_type == 'e-reserve')].copy()
s_storage_e_reserve.rename(columns={'energy_reserve_up_MW': 'reserve_up_MW', 'energy_reserve_down_MW': 'reserve_down_MW'}, inplace=True)
s_system_e_reserve = s_uc['energy_reserve'][s_uc['energy_reserve'].resource == 'system'].copy()
s_system_e_reserve.rename(columns={'required_energy_reserve_up_MW': 'required_reserve_up_MW', 'required_energy_reserve_down_MW': 'required_reserve_down_MW'}, inplace=True)
s_storage_dynamic = s_uc['reserve'][s_uc['reserve'].resource.isin(['battery','hydro_reservoir', 'CSP'])] #'hydro_reservoir', 'CSP'
s_storage_dynamic =  s_storage_dynamic[(s_storage_dynamic.model_type == 'envelope') & (s_storage_dynamic['µ'] == 'mu_2')]
s_storage_dynamic['model_type'] = 'dynamic'
s_system_dynamic = s_uc['reserve'][s_uc['reserve'].resource == 'system']
s_system_dynamic =  s_system_dynamic[(s_system_dynamic.model_type == 'envelope') & (s_system_dynamic['µ'] == 'mu_2')]
s_system_dynamic['model_type'] = 'dynamic'
s_storage_conservative = s_uc['reserve'][s_uc['reserve'].resource.isin(['battery','hydro_reservoir', 'CSP'])] #'hydro_reservoir', 'CSP'
s_storage_conservative =  s_storage_conservative[(s_storage_conservative.model_type == 'envelope') & (s_storage_conservative['µ'] == 'mu_1')]
s_storage_conservative['model_type'] = 'conservative'
s_system_conservative = s_uc['reserve'][s_uc['reserve'].resource == 'system']
s_system_conservative =  s_system_conservative[(s_system_conservative.model_type == 'envelope') & (s_system_conservative['µ'] == 'mu_1')]
s_system_conservative['model_type'] = 'conservative'


In [ ]:

select_ = ['model_type', 'day', 'hour', 'resource', 'reserve_up_MW', 'reserve_down_MW', 'required_reserve_up_MW', 'required_reserve_down_MW']
s_storage = pd.concat([s_storage_dynamic.filter(select_), s_storage_conservative.filter(select_), s_storage_e_reserve[s_storage_e_reserve.hour_i == s_storage_e_reserve.hour].filter(select_)], axis=0)
s_system = pd.concat([s_system_dynamic.filter(select_), s_system_conservative.filter(select_), s_system_e_reserve[s_system_e_reserve.hour_i == s_system_e_reserve.hour].filter(select_)], axis=0)


In [ ]:

indices_ = ['model_type', 'day']
s_storage_sum = s_storage.groupby(indices_)[['reserve_up_MW', 'reserve_down_MW']].sum()#.rename(columns={'reserve_up_MW': 'Up', 'reserve_down_MW': 'Down'})
s_storage_sum = add_up_down_multiindex(s_storage_sum).rename(columns = {'reserve_MW' : 'storage_reserve_commitment'})
s_system_sum = s_system.groupby(indices_)[['required_reserve_up_MW', 'required_reserve_down_MW']].sum()#.rename(columns={'required_reserve_up_MW': 'Up', 'required_reserve_down_MW': 'Down'})
s_system_sum = add_up_down_multiindex(s_system_sum).rename(columns = {'required_reserve_MW' : 'storage_reserve_commitment'})
s_storage_commitment = s_storage_sum / s_system_sum


In [ ]:
s_uc_storage = s_uc['storage']
s_ed_storage = s_ed['storage']
s_uc_storage['net_discharge_MW'] = s_uc_storage.discharge_MW - s_uc_storage.charge_MW
s_ed_storage['net_discharge_MW'] = s_ed_storage.discharge_MW - s_ed_storage.charge_MW

# Create model_type categorization without copying - use boolean masks
s_uc_e_reserve = s_uc_storage[s_uc_storage.model_type == 'e-reserve'][['model_type', 'day', 'hour', 'resource', 'r_id','net_discharge_MW']]
s_ed_e_reserve = s_ed_storage[s_ed_storage.model_type == 'e-reserve'][['model_type', 'day', 'hour', 'resource','r_id','iteration','net_discharge_MW']]

# Use masks to avoid multiple copies
uc_envelope = s_uc_storage.model_type == 'envelope'
ed_envelope = s_ed_storage.model_type == 'envelope'

s_uc_dynamic = s_uc_storage[uc_envelope & (s_uc_storage['µ']== 'mu_2')][['day', 'hour', 'resource', 'r_id','net_discharge_MW']].copy()
s_uc_dynamic['model_type'] = 'dynamic'
s_ed_dynamic = s_ed_storage[ed_envelope & (s_ed_storage['µ'] == 'mu_2')][['model_type', 'day', 'hour', 'resource','r_id','iteration','net_discharge_MW']].copy()
s_ed_dynamic['model_type'] = 'dynamic'

s_uc_conservative = s_uc_storage[uc_envelope & (s_uc_storage['µ'] == 'mu_1')][['day', 'hour', 'resource', 'r_id','net_discharge_MW']].copy()
s_uc_conservative['model_type'] = 'conservative'
s_ed_conservative = s_ed_storage[ed_envelope & (s_ed_storage['µ'] == 'mu_1')][['model_type', 'day', 'hour', 'resource','r_id','iteration','net_discharge_MW']].copy()
s_ed_conservative['model_type'] = 'conservative'

# Aggregate UC storage directly - group early to minimize memory
select_ = ['model_type', 'day', 'hour', 'resource', 'r_id','net_discharge_MW']
s_uc_storage = pd.concat([s_uc_e_reserve, s_uc_dynamic[select_], s_uc_conservative[select_]], ignore_index=False)
s_uc_storage = s_uc_storage.groupby(['model_type', 'day'])['net_discharge_MW'].sum()

# For ED storage: group by the detailed columns first, compute mean, then aggregate
select__ = ['model_type', 'day', 'hour', 'resource','r_id','iteration','net_discharge_MW']
s_ed_storage = pd.concat([s_ed_e_reserve, s_ed_dynamic[select_], s_ed_conservative[select_]], ignore_index=False)
# Group by detailed columns, take mean across iterations, then sum across resources
s_ed_storage = s_ed_storage.groupby(['model_type', 'day', 'hour', 'resource', 'r_id'])['net_discharge_MW'].mean().reset_index()
s_ed_storage = s_ed_storage.groupby(['model_type', 'day'])['net_discharge_MW'].sum()

# Clean up intermediate variables to free memory
del s_uc_e_reserve, s_ed_e_reserve, s_uc_dynamic, s_ed_dynamic, s_uc_conservative, s_ed_conservative, uc_envelope, ed_envelope

s_storage_reserve_activation = (s_ed_storage - s_uc_storage).rename('storage_reserve_activation_MW').to_frame()
s_storage_reserve_activation['storage_reserve_up_activation_MW'] = s_storage_reserve_activation['storage_reserve_activation_MW'].clip(lower=0)
s_storage_reserve_activation['storage_reserve_down_activation_MW'] = (-s_storage_reserve_activation['storage_reserve_activation_MW']).clip(lower=0)
s_storage_reserve_activation = add_up_down_multiindex(s_storage_reserve_activation[['storage_reserve_up_activation_MW','storage_reserve_down_activation_MW']]).rename(columns={'storage_reserve_activation_MW': 'storage_reserve_activation'})
s_storage_reserve_activation = s_storage_reserve_activation/s_system_sum.rename(columns={'storage_reserve_commitment': 'storage_reserve_activation'})

In [ ]:
s_storage_reserve = pd.merge(s_storage_commitment.reset_index(), s_storage_reserve_activation.reset_index(), on=['model_type', 'day', 'reserve_direction'])

In [ ]:
to_plot  = s_storage_reserve.reset_index().rename(columns={'storage_reserve_commitment': 'Storage Reserve Commitment', 'storage_reserve_activation': 'Storage Reserve Activation'})
# to_plot  = s_storage_commitment.reset_index().rename(columns={'storage_reserve_commitment': 'Storage Reserve Commitment'})
# to_plot['Batteries Reserve Commitment'] = to_plot['Storage Reserve Commitment']*1
# to_plot['Storage Reserve Activation'] = to_plot['Storage Reserve Activation']*1
to_plot['Storage Reserve Commitment'] = to_plot['Storage Reserve Commitment']*1
to_plot = to_plot.melt(
    id_vars=['day','model_type','reserve_direction'],
    value_vars=['Storage Reserve Commitment', 'Storage Reserve Activation'])


to_plot['value'] = to_plot['value'].clip(upper=1)
to_plot = to_plot.replace({'model_type': {'envelope': 'dynamic'}})
fig = px.box(
    to_plot,
    x='reserve_direction',
    y='value',
    color='model_type',
    facet_col='variable',
    category_orders={"model_type": ["conservative", "dynamic", "e-reserve", "stochastic"]},
    labels = {'value': 'Fraction of reserves', 'reserve_direction': '', 'model_type': 'model type'},
    boxmode="group",
)
update_background(fig,  legend_attr, dim)
# fig.update_yaxes(matches=None)
# fig.layout['yaxis2'].update(showticklabels=False)
# fig.update_yaxes(matches=True)
for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        # annotation.text = annotation.text.replace(" ($/day)", "<br>($/day)").replace(" ($/day)", "<br>($/day)").replace(" (%/day)", "<br>(%/day)").replace(" (h/day)", "<br>(h/day)")
        annotation.text = annotation.text.replace("variable=", "")
n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n))
fig.show()


In [ ]:
# Following groups all r_id together
s_battery_e_reserve = s_storage_e_reserve[s_storage_e_reserve.resource == 'battery']

s_battery_sum = s_battery_e_reserve.groupby(['day','hour_i', 'hour'])[['reserve_up_MW', 'reserve_down_MW']].sum().rename(columns={'reserve_up_MW': 'Up', 'reserve_down_MW': 'Down'})
s_system_sum =  s_system_e_reserve.groupby(['day','hour_i', 'hour'])[['required_reserve_up_MW', 'required_reserve_down_MW']].sum().rename(columns={'required_reserve_up_MW': 'Up', 'required_reserve_down_MW': 'Down'})
fraction = (s_battery_sum / s_system_sum).reset_index()
fraction['duration'] = fraction['hour'] - fraction['hour_i']+1
mean = fraction.groupby(['duration'])[['Up','Down']].mean().reset_index()


In [ ]:



# to_plot.rename(columns={'variable': {'energy_reserve_up_MW': 'test'}}, inplace=True)
fig= px.box(
    fraction.melt(id_vars=['duration'], value_vars=['Up', 'Down']),
    x = 'duration',
    y = 'value',
    facet_col = 'variable',
    boxmode="group",
    labels = {'value': 'Fraction of energy <br> reserves  by batteries', 'duration': 'Duration [h]'},
)
fig.add_scatter(
              y = mean['Up'],
              x = mean['duration'],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 1, row = 1,
)
fig.add_scatter(
              y = mean['Down'],
              x = mean['duration'],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 2, row = 1,
)

fig.add_annotation(
  x=1-.59,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)
fig.add_annotation(
  x=1-.01,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)

update_background(fig, legend_attr, dim, False)
fig.update_yaxes(matches=None)
fig.layout['yaxis2'].update(showticklabels=False)
for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace("variable=", "")

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n+50))
fig.show()

if G_save:
    fig.write_image("energy_reserves_by_batteries.pdf", width=dim[0], height=dim[1], scale=1)

In [ ]:
s_battery_dynamic = s_storage_dynamic[s_storage_dynamic.resource == 'battery']
s_battery_sum = s_battery_dynamic.groupby(['day','hour'])[['reserve_up_MW', 'reserve_down_MW']].sum().rename(columns={'reserve_up_MW': 'Up', 'reserve_down_MW': 'Down'})
s_system_sum = s_system_dynamic.groupby(['day','hour'])[['required_reserve_up_MW', 'required_reserve_down_MW']].sum().rename(columns={'required_reserve_up_MW': 'Up', 'required_reserve_down_MW': 'Down'})
fraction = (s_battery_sum / s_system_sum).reset_index()
# fraction['duration'] = fraction['hour'] - fraction['hour_i']+1
mean = fraction.groupby(['hour'])[['Up','Down']].mean().reset_index()

In [ ]:
x_axis = 'hour'
fig= px.box(
    fraction.melt(id_vars=[x_axis], value_vars=['Up', 'Down']),
    x = x_axis,
    y = 'value',
    facet_col = 'variable',
    boxmode="group",
    labels = {'value': 'Fraction of reserves by batteries'},
)
fig.add_scatter(
              y = mean['Up'],
              x = mean[x_axis],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 1, row = 1,
)
fig.add_scatter(
              y = mean['Down'],
              x = mean[x_axis],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 2, row = 1,
)

fig.add_annotation(
  x=0.03,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)
fig.add_annotation(
  x=1-.01,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)

update_background(fig, legend_attr, dim, False)
fig.update_yaxes(matches=None)
fig.layout['yaxis2'].update(showticklabels=False)
for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace("variable=", "")

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n+50))
fig.show()

if G_save:
    fig.write_image("reserves_by_batteries.pdf", width=dim[0], height=dim[1], scale=1)

In [ ]:

s_hydro_reservoir = s_storage_e_reserve[s_storage_e_reserve.resource == 'hydro_reservoir']

s_hydro_reservoir_sum = s_hydro_reservoir.groupby(['day','hour_i', 'hour'])[['reserve_up_MW', 'reserve_down_MW']].sum().rename(columns={'reserve_up_MW': 'Up', 'reserve_down_MW': 'Down'})
s_system_sum =  s_system_e_reserve.groupby(['day','hour_i', 'hour'])[['required_reserve_up_MW', 'required_reserve_down_MW']].sum().rename(columns={'required_reserve_up_MW': 'Up', 'required_reserve_down_MW': 'Down'})
fraction = (s_hydro_reservoir_sum / s_system_sum).reset_index()
fraction['duration'] = fraction['hour'] - fraction['hour_i']+1
mean = fraction.groupby(['duration'])[['Up','Down']].mean().reset_index()


In [ ]:



# to_plot.rename(columns={'variable': {'energy_reserve_up_MW': 'test'}}, inplace=True)
fig= px.box(
    fraction.melt(id_vars=['duration'], value_vars=['Up', 'Down']),
    x = 'duration',
    y = 'value',
    facet_col = 'variable',
    boxmode="group",
    labels = {'value': 'Fraction of energy <br> reserves  by hydro reservoir', 'duration': 'Duration [h]'},
)
fig.add_scatter(
              y = mean['Up'],
              x = mean['duration'],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 1, row = 1,
)
fig.add_scatter(
              y = mean['Down'],
              x = mean['duration'],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 2, row = 1,
)

fig.add_annotation(
  x=1-.59,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)
fig.add_annotation(
  x=1-.01,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)

update_background(fig, legend_attr, dim, False)
fig.update_yaxes(matches=None)
fig.layout['yaxis2'].update(showticklabels=False)
for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace("variable=", "")

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n+50))
fig.show()


In [ ]:
tuples = [(d, h_i, h) for d in s_battery_dynamic.day.unique() for h in s_battery_dynamic.hour.unique() for h_i in s_battery_dynamic.hour.unique() if h_i <= h]
s_battery_energy = pd.DataFrame(tuples, columns=['day', 'hour_i', 'hour'])   
s_battery_energy = s_battery_energy.merge(s_battery_dynamic, left_on=['day','hour'], right_on=['day','hour']) 
s_battery_energy.set_index(['day','hour_i','hour'], inplace=True)
# Following groups all batteries together
s_battery_energy_sum = s_battery_energy.groupby(['day','hour_i', 'hour'])[['reserve_up_MW', 'reserve_down_MW']].sum().rename(columns={'reserve_up_MW': 'Up', 'reserve_down_MW': 'Down'})
s_battery_energy_sum = s_battery_energy_sum.groupby(['day','hour_i']).cumsum()
# s_battery_energy_sum

s_system_energy = pd.DataFrame(tuples, columns=['day', 'hour_i', 'hour'])   
s_system_energy = s_system_energy.merge(s_system_dynamic, left_on=['day','hour'], right_on=['day','hour']) 
s_system_energy.set_index(['day','hour_i','hour'], inplace=True)
# Following groups all together (not needed in theory because there is just one system)
s_system_energy_sum = s_system_energy.groupby(['day','hour_i', 'hour'])[['required_reserve_up_MW', 'required_reserve_down_MW']].sum().rename(columns={'required_reserve_up_MW': 'Up', 'required_reserve_down_MW': 'Down'})
s_system_energy_sum = s_system_energy_sum.groupby(['day','hour_i']).cumsum()
# s_system_energy_sum

In [ ]:
fraction = (s_battery_energy_sum / s_system_energy_sum).reset_index()
fraction['duration'] = fraction['hour'] - fraction['hour_i']+1
mean = fraction.groupby(['duration'])[['Up','Down']].mean().reset_index()
# mean = fraction.groupby(['hour_i'])[['Up','Down']].mean().reset_index()

In [ ]:
# x_axis = 'hour_i'
x_axis = 'duration'
# x_axis = 'hour'
fig= px.box(
    fraction.melt(id_vars=[x_axis], value_vars=['Up', 'Down']),
    x = x_axis,
    y = 'value',
    facet_col = 'variable',
    boxmode="group",
    labels = {'value': 'Fraction of reserves by batteries', 'duration': 'Duration [h]'},
)
fig.add_scatter(
              y = mean['Up'],
              x = mean[x_axis],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 1, row = 1,
)
fig.add_scatter(
              y = mean['Down'],
              x = mean[x_axis],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 2, row = 1,
)

fig.add_annotation(
  x=1-.59,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)
fig.add_annotation(
  x=1-.01,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)

update_background(fig, legend_attr, dim, False)
fig.update_yaxes(matches=None)
fig.layout['yaxis2'].update(showticklabels=False)
for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace("variable=", "")

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n+50))
fig.show()


In [ ]:
s_hydro_reservoir_dynamic = s_storage_dynamic[s_storage_dynamic.resource == 'hydro_reservoir']
s_hydro_reservoir_sum = s_hydro_reservoir_dynamic.groupby(['day','hour'])[['reserve_up_MW', 'reserve_down_MW']].sum().rename(columns={'reserve_up_MW': 'Up', 'reserve_down_MW': 'Down'})

In [ ]:
tuples = [(d, h_i, h) for d in s_hydro_reservoir_dynamic.day.unique() for h in s_hydro_reservoir_dynamic.hour.unique() for h_i in s_hydro_reservoir_dynamic.hour.unique() if h_i <= h]
s_hydro_reservoir_energy = pd.DataFrame(tuples, columns=['day', 'hour_i', 'hour'])   
s_hydro_reservoir_energy = s_hydro_reservoir_energy.merge(s_hydro_reservoir_dynamic, left_on=['day','hour'], right_on=['day','hour']) 
s_hydro_reservoir_energy.set_index(['day','hour_i','hour'], inplace=True)
# Following groups all batteries together
s_hydro_reservoir_energy_sum = s_hydro_reservoir_energy.groupby(['day','hour_i', 'hour'])[['reserve_up_MW', 'reserve_down_MW']].sum().rename(columns={'reserve_up_MW': 'Up', 'reserve_down_MW': 'Down'})
s_hydro_reservoir_energy_sum = s_hydro_reservoir_energy_sum.groupby(['day','hour_i']).cumsum()
# s_battery_energy_sum

s_system_energy = pd.DataFrame(tuples, columns=['day', 'hour_i', 'hour'])   
s_system_energy = s_system_energy.merge(s_system_dynamic, left_on=['day','hour'], right_on=['day','hour']) 
s_system_energy.set_index(['day','hour_i','hour'], inplace=True)
# Following groups all together (not needed in theory because there is just one system)
s_system_energy_sum = s_system_energy.groupby(['day','hour_i', 'hour'])[['required_reserve_up_MW', 'required_reserve_down_MW']].sum().rename(columns={'required_reserve_up_MW': 'Up', 'required_reserve_down_MW': 'Down'})
s_system_energy_sum = s_system_energy_sum.groupby(['day','hour_i']).cumsum()
# s_system_energy_sum

In [ ]:
fraction = (s_hydro_reservoir_energy_sum / s_system_energy_sum).reset_index()
fraction['duration'] = fraction['hour'] - fraction['hour_i']+1
mean = fraction.groupby(['duration'])[['Up','Down']].mean().reset_index()
# mean = fraction.groupby(['hour_i'])[['Up','Down']].mean().reset_index()

In [ ]:
# x_axis = 'hour_i'
x_axis = 'duration'
# x_axis = 'hour'
fig= px.box(
    fraction.melt(id_vars=[x_axis], value_vars=['Up', 'Down']),
    x = x_axis,
    y = 'value',
    facet_col = 'variable',
    boxmode="group",
    labels = {'value': 'Fraction of reserves by hydro reservoirs', 'duration': 'Duration [h]'},
)
fig.add_scatter(
              y = mean['Up'],
              x = mean[x_axis],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 1, row = 1,
)
fig.add_scatter(
              y = mean['Down'],
              x = mean[x_axis],
              mode='markers',
              marker=dict(size=5, color = g_ORANGE, symbol="diamond"),
              col = 2, row = 1,
)

fig.add_annotation(
  x=1-.59,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)
fig.add_annotation(
  x=1-.01,
  y=1-.03,
  xref="paper",
  yref="paper",
  text="◆ mean",
  showarrow=False,
  font=dict(color=g_ORANGE),
)

update_background(fig, legend_attr, dim, False)
fig.update_yaxes(matches=None)
fig.layout['yaxis2'].update(showticklabels=False)
for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace("variable=", "")

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n+50))
fig.show()

# 

In [ ]:
# s_storage = s_uc['reserve'][s_uc['reserve'].resource.isin(['battery','hydro_reservoir', 'CSP'])] #
# s_battery =  s_storage[(s_storage.model_type == 'envelope') & (s_storage['µ'] == 'mu_2')]

# s_system = s_uc['reserve'][s_uc['reserve'].resource == 'system']
# s_system =  s_system[(s_system.model_type == 'envelope') & (s_system['µ'] == 'mu_2')]

In [ ]:
required_reserve = s_uc['reserve'][['hour','day','resource','µ','required_reserve_up_MW', 'required_reserve_down_MW']]
required_reserve = required_reserve[(required_reserve.resource == 'system')&(required_reserve['µ'] == 'mu_1')]
required_reserve.set_index(['day','hour'], inplace=True)
cum_required_reserve = required_reserve.groupby('day')[['required_reserve_up_MW', 'required_reserve_down_MW']].cumsum()
cum_required_reserve.rename(columns={'required_reserve_up_MW': 'Up', 'required_reserve_down_MW': 'Down'}, inplace=True)
cum_required_reserve['type'] = 'cumulative_reserve'

required_e_reserve = s_uc['energy_reserve'][['hour','hour_i','day','resource','required_energy_reserve_up_MW', 'required_energy_reserve_down_MW']]
required_e_reserve = required_e_reserve[(required_e_reserve.resource == 'system') & (required_e_reserve.hour_i == 1)]
required_e_reserve.set_index(['day','hour'], inplace=True)
required_e_reserve.rename(columns={'required_energy_reserve_up_MW': 'Up', 'required_energy_reserve_down_MW': 'Down'}, inplace=True)
required_e_reserve['type']= 'energy_reserve'

required_all = pd.concat([pd.melt(required_e_reserve.reset_index(), id_vars=['hour','day','type'], value_vars=['Up', 'Down']),
                pd.melt(cum_required_reserve.reset_index(), id_vars=['hour','day','type'], value_vars=['Up', 'Down'])])
required_day = required_all[required_all.day == 131]

In [ ]:
required_day = required_day.pivot(
    index='hour',
    columns=['type','variable'],
    values='value'
    )

In [ ]:
fig = px.box(required_all, x = 'hour', y = 'value', color = 'type', facet_col='variable', hover_data='day',boxmode="group", labels = {'value': 'energy requirements [MWh]'})

update_background(fig, legend_attr, dim, True)

fig.update_yaxes(matches=None)
fig.layout['yaxis2'].update(showticklabels=False)
for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace("variable=", "")

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n+50))
if G_save:
    fig.write_image("cumulative_energy_requirements.pdf", width=dim[0], height=dim[1], scale=1)
else:
    fig.show()

In [ ]:
day = 131
fig = px.line(required_all[required_all.day == day].sort_values('hour'), x='hour', y='value', color='type', facet_col='variable', symbol='type', labels = {'value': 'energy requirements [MWh]'})
update_background(fig, legend_attr, dim, True)

fig.update_yaxes(matches=None)
fig.layout['yaxis2'].update(showticklabels=False)
for annotation in fig.layout.annotations:
    if annotation.text.startswith("variable="):
        annotation.text = annotation.text.replace("variable=", "")

n = 4
fig.update_layout(margin=dict(l=n, r=n+30, t=n+30, b=n+50))
if G_save:
    fig.write_image(f"cumulative_energy_requirements_day_{day}.pdf", width=dim[0], height=dim[1], scale=1)
else:
    fig.show()